<a href="https://colab.research.google.com/github/cwilson-su/Large-Scale-Data-Analytics-Labs/blob/main/LAB3_2026_streaming.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP3 — Data streaming avec Spark Structured Streaming

**MU5IN852 — LSDA, 2026–2027** ·version du 05/11/2026

Ce TP accompagne le cours 3a *Data streaming et requêtes continues*. Il suit le même plan :

| Partie | Notions du cours | Durée indicative |
|---|---|---|
| Exercice 1 | fenêtres fixes et glissantes, calcul incrémental, état (*panes*) | 45 min |
| Exercice 2 | flux = table non bornée, modes de sortie Complete / Update / Append | 30 min |
| Exercice 3 | fenêtres sur le temps d'événement, données en retard, watermark | 60 min |
| Exercice 4 | jointures flux–table et flux–flux, purge de l'état, jointure externe | 45 min |

**Principe du TP : on contrôle le temps d'événement.** Les flux sont simulés par des fichiers CSV déposés un par un dans un dossier.
Chaque appel à `envoyer(...)` dépose **un lot** de tuples dont **nous choisissons les dates** (le temps d'événement),
puis attend que Spark l'ait traité. Le résultat est donc **reproductible** : vous pouvez prédire ce que Spark va faire,
puis vérifier.

Après chaque lot, `envoyer` affiche :
- le **watermark** utilisé pour traiter le lot ;
- le nombre de tuples **ignorés** car trop en retard ;
- le nombre de lignes conservées dans l'**état** ;
- ce qui est **écrit en sortie**.

## Partie 0 — Préparation (à exécuter, rien à écrire)

In [ ]:
# Installation de PySpark (environ 1 minute sur Google Colab).
# En local, si PySpark est déjà installé, cette cellule ne fait rien de plus.
!pip install -q pyspark

# PySpark 4 nécessite Java 17 ou plus récent : on l'installe si besoin.
import os, re, subprocess
try:
    sortie = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    version_java = int(re.search(r'version "(\d+)', sortie).group(1))
except Exception:
    version_java = 0
if version_java < 17:
    !apt-get install -y -qq openjdk-17-jdk-headless > /dev/null
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
print("Java", version_java if version_java >= 17 else 17)

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pandas as pd

spark = (SparkSession.builder
         .master("local[2]")
         .appName("TP3 streaming")
         .config("spark.sql.shuffle.partitions", "2")         # petites données : peu de partitions
         .config("spark.sql.session.timeZone", "UTC")         # les heures affichées sont celles que l'on écrit
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.sql.execution.arrow.pyspark.enabled", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

### Outils fournis pour simuler des flux

La cellule suivante définit quelques fonctions utilitaires. **Exécutez-la sans la lire** : son code n'est pas au programme,
seule son utilisation compte.

- `nouveau_flux(nom, schema)` crée un flux vide et renvoie le **DataFrame de streaming** qui le lit.
- `lancer(requete, mode)` démarre la requête continue avec le mode de sortie choisi (`"complete"`, `"update"` ou `"append"`).
- `envoyer(nom_flux=[tuples], ...)` dépose un lot dans un ou plusieurs flux, attend son traitement et affiche le résultat.
- `h("12:04")` construit la date `2026-01-15 12:04:00` (toutes les dates du TP sont le même jour).

Le nom donné à `nouveau_flux` doit être exactement celui utilisé dans `envoyer` :
après `clics = nouveau_flux("clics", ...)`, on envoie un lot avec `envoyer(clics=[...])`.

In [ ]:
# @title Outils fournis (à exécuter, inutile de lire le code)
import shutil

BASE = "/tmp/tp3_flux"
JOUR = "2026-01-15"

def h(hhmm):
    "Date du TP à l'heure hh:mm."
    return f"{JOUR} {hhmm}:00"

class _Flux:
    def __init__(self, nom, schema):
        self.dossier = f"{BASE}/{nom}"
        shutil.rmtree(self.dossier, ignore_errors=True)
        os.makedirs(self.dossier)
        self.schema, self.n = schema, 0
    def dataframe(self):
        return (spark.readStream.schema(self.schema)
                .option("maxFilesPerTrigger", 1)      # un fichier = un micro-batch
                .csv(self.dossier))
    def deposer(self, tuples):
        self.n += 1
        tmp = f"{self.dossier}/.lot_{self.n:03d}.tmp"   # fichier caché, ignoré par Spark
        with open(tmp, "w") as f:
            for t in tuples:
                f.write(",".join(str(x) for x in t) + "\n")
        os.rename(tmp, f"{self.dossier}/lot_{self.n:03d}.csv")   # apparition atomique

_FLUX, _SORTIES = {}, {}
_REQ, _DERNIER_LOT = None, -1

def nouveau_flux(nom, schema):
    "Crée un flux vide et renvoie le DataFrame de streaming correspondant."
    _FLUX[nom] = _Flux(nom, schema)
    return _FLUX[nom].dataframe()

def arreter():
    for q in spark.streams.active:
        q.stop()

def lancer(requete, mode):
    "Démarre la requête continue ; la sortie de chaque micro-batch est capturée."
    global _REQ, _DERNIER_LOT
    arreter()
    _SORTIES.clear()
    _DERNIER_LOT = -1
    def capturer(df_lot, num_lot):
        _SORTIES[num_lot] = df_lot.toPandas()
    _REQ = requete.writeStream.outputMode(mode).foreachBatch(capturer).start()
    print(f"Requête démarrée en mode {mode}.")

def _heure(iso):
    return "aucun" if (iso is None or iso.startswith("1970")) else iso[11:16]

def envoyer(**lots):
    "envoyer(flux1=[tuples], flux2=[tuples]) : dépose un lot par flux et affiche le traitement."
    global _DERNIER_LOT
    for nom, tuples in lots.items():
        _FLUX[nom].deposer(tuples)
    _REQ.processAllAvailable()
    progres = [p for p in _REQ.recentProgress if p["batchId"] > _DERNIER_LOT]
    if progres:
        _DERNIER_LOT = progres[-1]["batchId"]
    for nom, tuples in lots.items():
        print(f"Lot envoyé sur « {nom} » :", "  ".join(str(t).replace(f"{JOUR} ", "").replace(":00'", "'") for t in tuples))
    avec_donnees = [p for p in progres if p["numInputRows"] > 0]
    if avec_donnees:
        p = avec_donnees[0]
        wm = p.get("eventTime", {}).get("watermark")
        ignores = sum(s.get("numRowsDroppedByWatermark", 0) for s in p.get("stateOperators", []))
        print(f"   watermark utilisé : {_heure(wm)}    tuples ignorés (trop en retard) : {ignores}")
    if progres:
        etat = sum(s.get("numRowsTotal", 0) for s in progres[-1].get("stateOperators", []))
        print(f"   lignes dans l'état après le lot : {etat}")
    sorties = [_SORTIES[p["batchId"]] for p in progres
               if p["batchId"] in _SORTIES and len(_SORTIES[p["batchId"]]) > 0]
    if sorties:
        print("   écrit en sortie :")
        tableau = pd.concat(sorties).astype(object).fillna("NULL").to_string(index=False)
        print("\n".join("      " + l for l in tableau.split("\n")))
    else:
        print("   écrit en sortie : (rien)")
    print()

def hm(colonne):
    "Affiche une colonne date au format hh:mm."
    return F.date_format(colonne, "HH:mm")

print("Outils prêts.")

> **Lire l'affichage de `envoyer`.** Après chaque lot, quatre informations sont affichées :
> - **watermark utilisé** : celui avec lequel les tuples du lot ont été jugés. Il est calculé à partir des lots **précédents**,
>   jamais à partir du lot lui-même.
> - **tuples ignorés** : les tuples du lot rejetés car trop en retard par rapport à ce watermark.
> - **lignes dans l'état** : la taille de l'état **après** le lot, une fois le nouveau watermark appliqué.
> - **écrit en sortie** : tout ce que la requête a produit pour ce lot.
>
> **Attention :** quand un lot fait avancer le watermark, Spark applique aussitôt ce nouveau watermark (il ferme des fenêtres,
> purge l'état) **avant le lot suivant**. L'état et la sortie affichés en tiennent compte, alors que le watermark affiché est
> encore l'ancien. Le nouveau watermark apparaît à la ligne « watermark utilisé » du lot suivant.

---
## Exercice 1 — Fenêtres sur un flux, sans Spark

**Objectif :** comprendre ce que doit calculer et mémoriser un moteur de streaming, en l'implémentant soi-même.

On considère le flux **S1**(heure, v) ci-dessous. Dans cet exercice, on suppose que les tuples **arrivent dans l'ordre**
de leur heure. Les heures sont manipulées en minutes depuis minuit grâce aux fonctions `minutes` et `hhmm`.

In [ ]:
S1 = [("08:04", "A"), ("08:08", "B"), ("08:10", "B"), ("08:12", "C"), ("08:12", "B"),
      ("08:12", "A"), ("08:18", "C"), ("08:20", "B"), ("08:20", "B"), ("08:41", "A")]

def minutes(hhmm):
    "'08:04' -> 484"
    h, m = hhmm.split(":")
    return int(h) * 60 + int(m)

def hhmm(mn):
    "484 -> '08:04'"
    return f"{mn // 60:02d}:{mn % 60:02d}"

### Question 1 — Fenêtres fixes

La requête **F1** donne la fréquence de chaque valeur `v` dans chaque fenêtre fixe de 10 minutes.
La cellule suivante calcule le résultat attendu avec Spark, **sur les données statiques** (traitement batch).

In [ ]:
S1_df = spark.createDataFrame([(h(t), v) for t, v in S1], "heure string, v string") \
             .withColumn("heure", F.to_timestamp("heure"))

F1 = (S1_df.groupBy(F.window("heure", "10 minutes"), "v").count()
          .select(hm("window.start").alias("debut"), hm("window.end").alias("fin"), "v", "count")
          .orderBy("debut", "v"))
F1.show()

**1.a)** Complétez la classe `FenetresFixes` pour calculer F1 **au fil de l'eau** :
- la requête reçoit les tuples un par un avec `recevoir(t)` ;
- elle affiche le résultat d'une fenêtre dès qu'elle sait que cette fenêtre est terminée ;
- elle ne garde en mémoire (son **état**) que ce qui est nécessaire.

Attention : il peut y avoir des fenêtres **sans aucun tuple** (regardez le dernier tuple de S1).

In [ ]:
class FenetresFixes:
    def __init__(self, debut, taille):
        self.debut = minutes(debut)   # début de la première fenêtre, en minutes
        self.taille = taille          # taille d'une fenêtre, en minutes
        self.courante = 0             # numéro de la fenêtre en cours
        self.freq = {}                # état : fréquences dans la fenêtre en cours

    def recevoir(self, t):
        heure, v = t
        # À COMPLÉTER :
        # 1) calculer le numéro k de la fenêtre du tuple
        # 2) si k dépasse la fenêtre courante, émettre la ou les fenêtres terminées
        # 3) mettre à jour l'état
        ...

    def emettre(self):
        "Affiche le résultat de la fenêtre courante."
        d = self.debut + self.courante * self.taille
        for v in sorted(self.freq):
            print(f"[{hhmm(d)}, {hhmm(d + self.taille)}[  v={v}  nb={self.freq[v]}")

    def terminer(self):
        "Fin du flux : émet la dernière fenêtre."
        self.emettre()


requete = FenetresFixes("08:00", 10)
for t in S1:
    requete.recevoir(t)
requete.terminer()

**1.b)** Quelle est la taille de l'état de votre requête ? Dépend-elle du nombre de tuples reçus ?

*Votre réponse :*

### Question 2 — Fenêtres glissantes et calcul incrémental

La requête **F2** donne la fréquence de chaque valeur dans des fenêtres de **10 minutes**, avec un **décalage de 5 minutes**.

In [ ]:
F2 = (S1_df.groupBy(F.window("heure", "10 minutes", "5 minutes"), "v").count()
          .select(hm("window.start").alias("debut"), hm("window.end").alias("fin"), "v", "count")
          .where("debut >= '08:00'")          # on ignore la fenêtre [07:55, 08:05[
          .orderBy("debut", "v"))
F2.show(30)

Comme dans le cours (diapo *Approche 2 : un état par intervalle*), on découpe le temps en **intervalles** $w_k$
de la durée du décalage, à partir de 08:00, et on note $s_k$ le résultat partiel (les fréquences) de l'intervalle $w_k$.

**2.a)** La taille d'une fenêtre est un multiple du décalage. Combien d'intervalles une fenêtre contient-elle ?

*Votre réponse :*

**2.b)** Donnez $s_0$, $s_1$ et $s_2$ pour $w_0 = [08{:}00, 08{:}05[$, $w_1 = [08{:}05, 08{:}10[$, $w_2 = [08{:}10, 08{:}15[$.

*Votre réponse :*
- $s_0$ = ...
- $s_1$ = ...
- $s_2$ = ...

**2.c)** Exprimez les résultats $R_0$ et $R_1$ des fenêtres $W_0 = [08{:}00, 08{:}10[$ et $W_1 = [08{:}05, 08{:}15[$
à partir des $s_k$, puis donnez leur valeur.

*Votre réponse :*
- $R_0$ = ...
- $R_1$ = ...

**2.d)** Complétez la classe `FenetresGlissantes` qui calcule F2 incrémentalement :
- chaque tuple met à jour **un seul** résultat partiel, celui de son intervalle ;
- le résultat d'une fenêtre est obtenu en combinant les résultats partiels de ses intervalles ;
- une fenêtre est émise dès qu'un tuple arrive dans un intervalle situé **après** sa fin.

Dans un premier temps, on garde tous les résultats partiels (`purger=False`).

**2.e)** Ajoutez ensuite la **purge** : quand `purger=True`, supprimez de l'état les résultats partiels qui ne serviront
plus à aucune fenêtre future. L'attribut `max_etat` mesure le nombre maximal d'intervalles conservés.

In [ ]:
class FenetresGlissantes:
    def __init__(self, debut, taille, decalage, purger=False):
        assert taille % decalage == 0, "la taille doit être un multiple du décalage"
        self.debut = minutes(debut)
        self.taille, self.decalage = taille, decalage
        self.n = taille // decalage   # nombre d'intervalles par fenêtre
        self.intervalles = {}         # état : numéro d'intervalle -> fréquences partielles s_k
        self.prochaine = 0            # numéro de la prochaine fenêtre à émettre
        self.purger = purger
        self.max_etat = 0             # nombre maximal d'intervalles conservés

    def recevoir(self, t):
        heure, v = t
        p = (minutes(heure) - self.debut) // self.decalage   # intervalle du tuple
        # À COMPLÉTER :
        # 1) émettre les fenêtres terminées (la fenêtre i couvre les intervalles i .. i+n-1)
        #    et, si self.purger, supprimer les intervalles devenus inutiles
        # 2) mettre à jour le résultat partiel de l'intervalle p
        ...
        self.max_etat = max(self.max_etat, len(self.intervalles))

    def emettre(self, i):
        "Combine les résultats partiels des intervalles de la fenêtre i et l'affiche."
        # À COMPLÉTER
        ...

    def terminer(self):
        "Fin du flux : émet les fenêtres restantes."
        dernier = max(self.intervalles)
        while self.prochaine <= dernier:
            self.emettre(self.prochaine)
            if self.purger:
                self.intervalles.pop(self.prochaine, None)
            self.prochaine += 1


for purger in [False, True]:
    print(f"--- purger={purger}")
    requete = FenetresGlissantes("08:00", 10, 5, purger)
    for t in S1:
        requete.recevoir(t)
    requete.terminer()
    print("nombre maximal d'intervalles conservés :", requete.max_etat)

**2.f)** Avec la purge, combien d'intervalles faut-il conserver au plus ? Comparez avec l'approche
« un état par fenêtre » vue en cours : combien de mises à jour par tuple dans chaque approche ?

*Votre réponse :*

### Question 3 — Et si les tuples arrivaient dans le désordre ?

Supposez que le tuple `("08:07", "A")` arrive **après** `("08:12", "C")`.

**3.a)** Que fait votre classe `FenetresGlissantes` (avec purge) ? Le résultat est-il correct ?

**3.b)** Que faudrait-il changer pour accepter des tuples en retard ? Quel est le prix à payer ?

*Votre réponse :*

---
## Exercice 2 — Un flux est une table non bornée : les modes de sortie

**Objectif :** observer ce que Spark écrit en sortie à chaque micro-batch selon le mode de sortie.

On reprend le *word count* du cours (diapo *Mode Complete : exemple du word count*) :
un flux `mots(ts, mot)` et trois lots.

In [ ]:
SCHEMA_MOTS = "ts timestamp, mot string"

def scenario_word_count():
    envoyer(mots=[(h("12:01"), "cat"), (h("12:01"), "dog"), (h("12:01"), "dog"), (h("12:01"), "dog")])
    envoyer(mots=[(h("12:02"), "owl"), (h("12:02"), "cat")])
    envoyer(mots=[(h("12:03"), "dog"), (h("12:03"), "owl")])

### Question 1 — Mode Complete

**1.a)** Écrivez la requête `compte` qui compte le nombre d'occurrences de chaque mot, **sans fenêtre**.
Remarquez qu'elle s'écrit exactement comme sur un DataFrame statique.

In [ ]:
mots = nouveau_flux("mots", SCHEMA_MOTS)

compte = ...   # À COMPLÉTER

lancer(compte, "complete")
scenario_word_count()

**1.b)** Comparez avec la figure du cours. Que contient la sortie à chaque lot ? Pourquoi le nombre de lignes de l'état augmente-t-il ?

*Votre réponse :*

### Question 2 — Mode Update

**2.a)** Relancez la même requête en mode `update`. Avant d'exécuter, prédisez ce qui sera écrit au lot 2 et au lot 3.

In [ ]:
mots = nouveau_flux("mots", SCHEMA_MOTS)
compte = mots.groupBy("mot").count()

lancer(compte, ...)   # À COMPLÉTER
scenario_word_count()

*Votre réponse :*

### Question 3 — Mode Append

**3.a)** Essayez le mode `append`. Que se passe-t-il ? Expliquez à l'aide du tableau *Quels modes de sortie pour quelle requête ?* du cours.

In [ ]:
mots = nouveau_flux("mots", SCHEMA_MOTS)
compte = mots.groupBy("mot").count()

try:
    lancer(compte, "append")
except Exception as e:
    print(type(e).__name__, ":", str(e)[:250])

*Votre réponse :*

**3.b)** Écrivez maintenant une requête **sans agrégation** : les tuples dont le mot est différent de `owl`.
Exécutez-la en mode `append`, puis essayez le mode `complete`. Expliquez.

In [ ]:
mots = nouveau_flux("mots", SCHEMA_MOTS)
filtre = ...   # À COMPLÉTER

lancer(filtre, "append")
scenario_word_count()

try:
    lancer(filtre, "complete")
except Exception as e:
    print(type(e).__name__, ":", str(e)[:250])

*Votre réponse :*

---
## Exercice 3 — Fenêtres sur le temps d'événement et watermark

**Objectif :** prédire puis vérifier le comportement d'une agrégation par fenêtre quand des données arrivent en retard.

On compte les mots par **fenêtre fixe de 10 minutes**. Le scénario comporte 7 lots ; certains tuples arrivent en retard
(leur date est antérieure à des dates déjà vues). Il prolonge l'exercice *Testez-vous : watermark* du cours.

In [ ]:
LOTS_EX3 = [
    [("12:02", "cat"), ("12:08", "dog"), ("12:13", "owl")],   # lot 1
    [("12:09", "cat"), ("12:07", "dog"), ("12:16", "owl")],   # lot 2
    [("12:04", "dog"), ("12:18", "cat")],                     # lot 3
    [("12:12", "dog"), ("12:26", "cat")],                     # lot 4
    [("12:19", "owl")],                                       # lot 5
    [("12:36", "owl")],                                       # lot 6
    [("12:40", "dog")],                                       # lot 7
]

def scenario_ex3():
    for lot in LOTS_EX3:
        envoyer(mots=[(h(t), m) for t, m in lot])

def par_fenetre(df):
    "Présentation lisible du résultat d'une agrégation par fenêtre."
    return df.select(hm("window.start").alias("debut"), hm("window.end").alias("fin"), "mot", "count")

### Question 1 — Sans watermark

**1.a)** Écrivez la requête `fenetres` qui compte les mots par fenêtre fixe de 10 minutes (utilisez `F.window`
et terminez par `par_fenetre(...)`), puis exécutez-la en mode `update`.

In [ ]:
mots = nouveau_flux("mots", SCHEMA_MOTS)
fenetres = ...   # À COMPLÉTER

lancer(fenetres, "update")
scenario_ex3()

**1.b)** Que devient le tuple `(12:04, dog)` du lot 3 ? Comment évolue le nombre de lignes dans l'état ?

*Votre réponse :*

### Question 2 — Avec un watermark : prédire avant d'exécuter

On ajoute `withWatermark("ts", "5 minutes")`. Rappels du cours :
- le watermark utilisé pour traiter le lot $i$ vaut $\mathit{wm}_i = d^{\max}_{i-1} - \text{retard}$,
  où $d^{\max}_{i-1}$ est la plus grande date vue **jusqu'au lot précédent** ;
- une fenêtre est **ouverte** tant que sa fin est $> \mathit{wm}_i$ ;
- un tuple est agrégé si sa fenêtre est encore ouverte, sinon il est ignoré.

**2.a)** Complétez le tableau **sans exécuter de code**.

| Lot | Tuples | $d^{\max}$ avant le lot | $\mathit{wm}$ utilisé | Tuples ignorés |
|---|---|---|---|---|
| 1 | 12:02 cat, 12:08 dog, 12:13 owl | — | aucun | |
| 2 | 12:09 cat, 12:07 dog, 12:16 owl | 12:13 | | |
| 3 | 12:04 dog, 12:18 cat | | | |
| 4 | 12:12 dog, 12:26 cat | | | |
| 5 | 12:19 owl | | | |
| 6 | 12:36 owl | | | |
| 7 | 12:40 dog | | | |

**2.b)** Écrivez la requête avec watermark, exécutez-la en mode `update` et vérifiez votre tableau.

In [ ]:
mots = nouveau_flux("mots", SCHEMA_MOTS)
fenetres_wm = ...   # À COMPLÉTER

lancer(fenetres_wm, "update")
scenario_ex3()

**2.c)** Comparez le nombre de lignes dans l'état avec la question 1. Pourquoi est-il maintenant borné ?

*Votre réponse :*

### Question 3 — Mode Append : quand une fenêtre est-elle écrite ?

**3.a)** Prédisez après quel lot chaque fenêtre ([12:00, 12:10[, [12:10, 12:20[, [12:20, 12:30[, …) sera écrite en mode `append`,
et avec quelles valeurs. Puis exécutez.

In [ ]:
mots = nouveau_flux("mots", SCHEMA_MOTS)
fenetres_wm = ...   # même requête qu'en 2.b

lancer(fenetres_wm, "append")
scenario_ex3()

*Votre réponse :*

### Question 4 — Choisir le retard toléré

**4.a)** Reprenez le tableau de la question 2 avec un retard de **10 minutes** : quels tuples sont maintenant pris en compte ?
Exécutez en mode `update` pour vérifier.

In [ ]:
mots = nouveau_flux("mots", SCHEMA_MOTS)
fenetres_wm10 = ...   # À COMPLÉTER

lancer(fenetres_wm10, "update")
scenario_ex3()

*Votre réponse :*

**4.b)** Résumez le compromis à faire pour choisir le retard toléré.

*Votre réponse :*

### Question 5 — Fenêtres glissantes : la règle s'applique fenêtre par fenêtre

Fenêtres de 10 minutes avec un décalage de 5 minutes, retard toléré de 5 minutes. Le scénario a deux lots :
`(12:11, x)` puis `(12:04, a)`.

**5.a)** À quelles fenêtres le tuple `(12:04, a)` appartient-il ? Avec quel watermark est-il traité ? Dans quelles fenêtres sera-t-il compté ?

**5.b)** Écrivez la requête et vérifiez.

*Votre réponse :*

In [ ]:
mots = nouveau_flux("mots", SCHEMA_MOTS)
glissantes = ...   # À COMPLÉTER

lancer(glissantes, "update")
envoyer(mots=[(h("12:11"), "x")])
envoyer(mots=[(h("12:04"), "a")])

---
## Exercice 4 — Jointures

**Objectif :** comprendre ce que Spark doit mémoriser pour joindre des flux, et quand il peut l'oublier.

On reprend l'exemple du cours : des **impressions** publicitaires et des **clics**.

In [ ]:
SCHEMA_IMP = "impAdId string, impTime timestamp"
SCHEMA_CLK = "clickAdId string, clickTime timestamp"

annonces = spark.createDataFrame([("a1", "Acme"), ("a2", "Globex"), ("a3", "Initech")],
                                 "adId string, annonceur string")
annonces.show()

### Question 1 — Jointure entre un flux et une table statique

**1.a)** Joignez le flux de clics à la table `annonces` (jointure interne sur `clickAdId = adId`), puis exécutez
le lot fourni. Faites de même avec une jointure externe `left` où le flux est à gauche.

In [ ]:
clics = nouveau_flux("clics", SCHEMA_CLK)
interne = ...   # À COMPLÉTER
lancer(interne, "append")
envoyer(clics=[("a1", h("10:00")), ("a9", h("10:01"))])

clics = nouveau_flux("clics", SCHEMA_CLK)
externe = ...   # À COMPLÉTER
lancer(externe, "append")
envoyer(clics=[("a1", h("10:00")), ("a9", h("10:01"))])

**1.b)** Pourquoi le clic `a9` sans annonce peut-il être émis avec `NULL` **immédiatement** ? Pourquoi l'état est-il vide ?

*Votre réponse :*

**1.c)** Essayez maintenant `annonces LEFT JOIN clics` (la table à gauche). Expliquez l'erreur.

In [ ]:
clics = nouveau_flux("clics", SCHEMA_CLK)
try:
    lancer(annonces.join(clics, F.expr("clickAdId = adId"), "left"), "append")
except Exception as e:
    print(type(e).__name__, ":", str(e)[:200])

*Votre réponse :*

### Question 2 — Jointure interne entre deux flux, sans watermark

On joint chaque impression aux clics sur la même annonce survenus **dans l'heure qui suit** l'impression.
Le scénario comporte 6 lots, chacun pouvant contenir des impressions et des clics.

In [ ]:
CONDITION = "clickAdId = impAdId AND clickTime >= impTime AND clickTime <= impTime + interval 1 hour"

def presenter(df):
    return df.select("impAdId", hm("impTime").alias("impTime"), "clickAdId", hm("clickTime").alias("clickTime"))

def scenario_ex4():
    envoyer(imp=[("a1", h("10:00")), ("a2", h("10:10")), ("a3", h("10:20"))], clk=[("a1", h("10:30"))])   # lot 1
    envoyer(imp=[("a4", h("11:00"))], clk=[("a2", h("11:05"))])                                          # lot 2
    envoyer(clk=[("a3", h("11:30"))])                                                                     # lot 3
    envoyer(imp=[("a5", h("14:00"))], clk=[("a5", h("14:30"))])                                          # lot 4
    envoyer(imp=[("a6", h("14:40"))])                                                                     # lot 5
    envoyer(imp=[("a7", h("15:30"))], clk=[("a7", h("15:50"))])                                          # lot 6

**2.a)** Prédisez les paires (impression, clic) produites à chaque lot. Écrivez la requête **sans watermark**,
exécutez-la en mode `append` et vérifiez.

In [ ]:
imp = nouveau_flux("imp", SCHEMA_IMP)
clk = nouveau_flux("clk", SCHEMA_CLK)
jointure = ...   # À COMPLÉTER (utilisez CONDITION et presenter)

lancer(jointure, "append")
scenario_ex4()

**2.b)** Le clic `a2` du lot 2 joint une impression arrivée au lot 1. Qu'est-ce que cela montre ?
Pourquoi le clic `a3` ne produit-il rien ? Comment évolue l'état ?

*Votre réponse :*

### Question 3 — Borner l'état avec des watermarks

On ajoute un watermark de **2 heures** sur les impressions et de **3 heures** sur les clics.
Le watermark de l'opérateur est le **minimum** des deux : $\mathit{wm} = \min(d^{\max}_{imp} - 2\,h,\ d^{\max}_{clk} - 3\,h)$.
D'après le cours, une impression de date $t$ est purgée quand $\mathit{wm} > t + 1\,h$, et un clic de date $c$ quand $\mathit{wm} > c$.

**3.a)** Complétez le tableau sans exécuter de code.

| Lot | $d^{\max}_{imp}$ avant le lot | $d^{\max}_{clk}$ avant le lot | $\mathit{wm}$ utilisé |
|---|---|---|---|
| 1 | — | — | aucun |
| 2 | 10:20 | 10:30 | |
| 3 | | | |
| 4 | | | |
| 5 | | | |
| 6 | | | |

Quels tuples sont purgés juste après le lot 4 ?

**3.b)** Écrivez la requête avec watermarks et vérifiez votre tableau (watermark utilisé, lignes dans l'état).

In [ ]:
imp = nouveau_flux("imp", SCHEMA_IMP)
clk = nouveau_flux("clk", SCHEMA_CLK)
jointure_wm = ...   # À COMPLÉTER

lancer(jointure_wm, "append")
scenario_ex4()

**3.c)** Les paires produites sont-elles les mêmes qu'à la question 2 ? Qu'est-ce qui a changé ?

*Votre réponse :*

### Question 4 — Jointure externe

**4.a)** On veut aussi les impressions **sans clic**. Prédisez **après quel lot** les paires `(a3, NULL)` et `(a4, NULL)`
seront émises, puis exécutez la jointure `leftOuter`.

In [ ]:
imp = nouveau_flux("imp", SCHEMA_IMP)
clk = nouveau_flux("clk", SCHEMA_CLK)
externe = ...   # À COMPLÉTER

lancer(externe, "append")
scenario_ex4()

*Votre réponse :*

**4.b)** Que se passe-t-il si on retire les watermarks de la jointure externe ? Expliquez.

In [ ]:
imp = nouveau_flux("imp", SCHEMA_IMP)
clk = nouveau_flux("clk", SCHEMA_CLK)
try:
    lancer(presenter(imp.join(clk, F.expr(CONDITION), "leftOuter")), "append")
except Exception as e:
    print(type(e).__name__, ":", str(e)[:200])

*Votre réponse :*

---
## Bilan

Répondez en une ou deux phrases.

1. Pourquoi une agrégation sans watermark ne peut-elle pas être exécutée en mode Append ?
2. Dans une agrégation par fenêtre, qu'est-ce qui décide si un tuple en retard est compté ?
3. Pourquoi faut-il **à la fois** un watermark et un prédicat temporel pour purger l'état d'une jointure entre deux flux ?
4. D'où vient la latence d'une jointure externe entre deux flux ?

*Vos réponses :*

In [ ]:
arreter()